In [1]:
import sys
import os

sys.path.append(os.path.abspath(r'C:\Users\esman\Documents\Github\mediguard\data\raw\drug_reference'))

# Eğer iki üst klasöre çıkmanız gerekiyorsa:
# sys.path.append(os.path.abspath('../..'))

# Artık üst klasördeki modülünüzü içe aktarabilirsiniz

In [2]:
import pandas as pd
import drug_categories  as drug

In [3]:
df= pd.read_csv("C:\\Users\\esman\\Documents\\Github\\mediguard\\data\\raw\\drug_reference\\titck_drugs_extracted.csv", encoding='utf-8')

In [4]:
df.head()

,drug_name,active_ingredient,category
0,"DUBLEVENT 0,5 MG + 2,5 MG / 2,5 ML NEBÜLİZASYO...","İPRATROPİUM BROMÜR, SALBUTAMOL SÜLFAT",Solunum
1,DEXPASS 50/300 MG FİLM KAPLI TABLET,"DEKSKETOPROFEN TROMETAMOL, PARASETAMOL",Ağrı Kesici
2,DEXPASS 25 MG/300 MG EFERVESAN TABLET,"DEKSKETOPROFEN TROMETAMOL, PARASETAMOL",Ağrı Kesici
3,PAXLOVID 150 MG/100 MG FILM KAPLI TABLET(ÜRETİ...,"NİRMATRELVİR, RİTONAVİR",Antiviral
4,AVELOX 400 MG FİLM KAPLI TABLET,MOKSİFLOKSASİN,Antibiyotik


In [5]:
def enrich_drug_data(row):
    drug_name = str(row.get('drug_name', '')).upper()
    active_ing = str(row.get('active_ingredient', '')).upper()
    
    # 1. Kategori Eşleştirme
    category = "Diğer / Sınıflandırılamadı"
    for keyword, cat in drug.CATEGORY_MAP.items():
        if keyword in active_ing:
            category = cat
            break
            
    # 2. Formülasyon Tespiti
    form = "Diğer"
    if any(kw in drug_name for kw in ["ŞURUP", "SÜSPANSİYON", "SUSPANSIYON", "DAMLA"]):
        form = "Şurup/Süspansiyon/Damla"
    elif "TABLET" in drug_name or "DRAJE" in drug_name:
        form = "Tablet"
    elif "KAPSÜL" in drug_name or "KAPSUL" in drug_name:
        form = "Kapsül"
    elif any(kw in drug_name for kw in ["ENJEKSİYON", "ENJEKSIYON", "FLAKON", "AMPUL", "ÇÖZELTİ", "COZELTI", "LİYOFİLİZE"]):
        # Solunum yolları için olan çözeltileri ayır
        if any(kw in drug_name for kw in ["NEBÜL", "NEBUL", "İNHAL", "INHAL"]):
            form = "İnhaler/Nebül"
        else:
            form = "Enjeksiyon/Flakon"
    elif any(kw in drug_name for kw in ["KREM", "MERHEM", "JEL", "POMAD"]):
        form = "Topikal (Krem/Jel)"
    elif "SUPOZİTUVAR" in drug_name or "SUPOZITUVAR" in drug_name or "FİTİL" in drug_name:
        form = "Supozituvar"
    elif any(kw in drug_name for kw in ["NEBÜL", "NEBUL", "İNHAL", "INHAL", "AEROSOL"]):
        form = "İnhaler/Nebül"

    # 3. Pediatrik True/False Etiketleme
    pediatric_keywords = ["ŞURUP", "SÜSPANSİYON", "SUSPANSIYON", "PEDİATRİK", "SÜSP"]
    is_pediatric = any(kw in drug_name for kw in pediatric_keywords)
    
    # 4. Kronik True/False Etiketleme
    dosage_type = drug.DOSAGE_TYPE_MAP.get(category, "")
    is_chronic = (dosage_type == "TIP2_KRONIK")
    
    return pd.Series([category, is_pediatric, is_chronic, form])

In [6]:
def process_and_save(input_file="C:\\Users\\esman\\Documents\\Github\\mediguard\\data\\titck_drugs_extracted.csv", output_file="C:\\Users\\esman\\Documents\\Github\\mediguard\\data\\raw\\processed\\titck_enriched_clean.csv"):
    print("Veriler yükleniyor...")
    df = pd.read_csv(input_file)
    
    print("Yeniden sınıflandırma ve zenginleştirme uygulanıyor...")
    # Yeni 'form' kolonu listeye eklendi
    df[['category', 'is_pediatric', 'is_chronic', 'form']] = df.apply(enrich_drug_data, axis=1)
    
    # Risk analizine girmeyecek olan "Diğer" grubunu çıkart
    clean_df = df[df['category'] != "Diğer / Sınıflandırılamadı"].copy()
    
    clean_df.to_csv(output_file, index=False, encoding="utf-8")
    print("\n--- İŞLEM ÖZETİ ---")
    print(f"Başlangıçtaki Toplam İlaç: {len(df)}")
    print(f"Filtrelenmiş ve Zenginleştirilmiş İlaç: {len(clean_df)}")
    print(f"Çıktı Kaydedildi: {output_file}")

In [7]:
process_and_save()

Veriler yükleniyor...
Yeniden sınıflandırma ve zenginleştirme uygulanıyor...

--- İŞLEM ÖZETİ ---
Başlangıçtaki Toplam İlaç: 89542
Filtrelenmiş ve Zenginleştirilmiş İlaç: 20194
Çıktı Kaydedildi: C:\Users\esman\Documents\Github\mediguard\data\raw\processed\titck_enriched_clean.csv
